# Day 14: Train the CNN-ViT Hybrid Fusion Network 🛰️

## 1. Objective and Engineering Context
On **Day 8**, we implemented the baseline `Hyperspectral3DCNN` to extract localized spatial-spectral textures with continuous 3D convolutions. On **Day 12**, we implemented `HyperspectralViT` using spatial patch tokenization and multi-head self-attention to capture global context across all 49 patch pixels. On **Day 13**, we constructed the `HyperspectralHybridCNNViT` dual-stream fusion architecture.

Today (**Day 14**), we implement and execute the **end-to-end training and regularization pipeline** for the Hybrid model:
1. Connect the real Salinas PCA-reduced DataLoaders (Day 9) to the Day 13 CNN-ViT Hybrid Network.
2. Address the **overfitting challenge**: The hybrid model contains **103,000 parameters** (~4.4x more than the 23,352 parameters of the 3D-CNN), operating on a benchmark dataset of 37,891 training pixels.
3. Apply rigorous regularization: **AdamW with decoupled weight decay** ($10^{-4}$) and **active dropout ($p=0.2$)** in the fusion projection head.
4. Verify dual-branch gradient backpropagation: Gradients flow simultaneously back into both the 3D-CNN and ViT transformer encoders.
5. Save the best hybrid model checkpoint atomically to `outputs/checkpoints/best_hybrid_cnn_vit.pt` without altering Day 10's `best_3d_cnn.pt`.
6. Reload the checkpoint into a fresh model instance and verify deterministic forward inference.

> **Engineering Note on Compute & Epochs**:
> Training a 103,000-parameter Transformer-CNN hybrid on CPU requires noticeable computation per epoch (297 batches of batch size 128 per epoch). To maintain scientific integrity without faking numbers, we execute a genuine **2-epoch training run** on the full real dataset, log exact real-time metrics, plot genuine diagnostic curves, and verify checkpoint reloading.

In [1]:
import sys
import time
from pathlib import Path

# Resolve project root dynamically
current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from terraspectra.data import (
    SalinasLoader,
    SALINAS_CLASS_NAMES,
    create_dataloaders,
)
from terraspectra.models import (
    Hyperspectral3DCNN,
    Hyperspectral3DCNNConfig,
    HyperspectralHybridCNNViT,
    HyperspectralHybridConfig,
    TrainingConfig,
    train_one_epoch,
    validate_one_epoch,
    train_model,
    save_checkpoint,
    load_checkpoint,
    select_device,
    set_seed,
)

print(f"Project Root:    {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")
device = select_device("auto")
print(f"Selected Device: {device}")
set_seed(42)

Project Root:    C:\Users\abhis\Desktop\Tetraspectra
PyTorch Version: 2.14.1+cpu
Selected Device: cpu


## 2. Parameter Comparison: Baseline 3D-CNN vs. CNN-ViT Hybrid

A critical ML engineering principle is understanding model complexity relative to data scale:
- The **Salinas ground truth** provides 54,129 labeled pixels across 16 classes.
- With a 70/15/15 split, our training set contains **37,891 patch samples**.
- Let us compare the exact parameter counts of the baseline 3D-CNN and the CNN-ViT Hybrid network.

In [2]:
# 1. Baseline 3D-CNN
cnn_config = Hyperspectral3DCNNConfig(
    in_channels=1,
    spectral_depth=15,
    spatial_size=7,
    num_classes=16,
    conv_channels=(8, 16, 32),
    kernel_spectral=(7, 5, 3),
    dropout_rate=0.3,
)
baseline_cnn = Hyperspectral3DCNN(cnn_config)
cnn_params = sum(p.numel() for p in baseline_cnn.parameters() if p.requires_grad)

# 2. CNN-ViT Hybrid Fusion Network
hybrid_config = HyperspectralHybridConfig(
    in_channels=1,
    spectral_depth=15,
    spatial_size=7,
    num_classes=16,
    cnn_conv_channels=(8, 16, 32),
    cnn_kernel_spectral=(7, 5, 3),
    vit_embed_dim=64,
    vit_num_heads=4,
    vit_num_layers=2,
    vit_mlp_dim=128,
    vit_pooling="mean",
    fusion_dim=64,
    dropout_rate=0.2,
)
hybrid_model = HyperspectralHybridCNNViT(hybrid_config)
hybrid_param_info = hybrid_model.count_parameters()
hybrid_params = hybrid_param_info["trainable_parameters"]

print("=" * 68)
print("ARCHITECTURAL COMPLEXITY COMPARISON")
print("=" * 68)
print(f"{'Component':<35} | {'3D-CNN':>12} | {'CNN-ViT Hybrid':>14}")
print("-" * 68)
print(f"{'Total Trainable Parameters':<35} | {cnn_params:>12,} | {hybrid_params:>14,}")
print(f"{'  - 3D-CNN Stream':<35} | {cnn_params:>12,} | {hybrid_param_info['component_breakdown']['cnn_branch_parameters']:>14,}")
print(f"{'  - ViT Transformer Stream':<35} | {'N/A':>12} | {hybrid_param_info['component_breakdown']['vit_branch_parameters']:>14,}")
print(f"{'  - Fusion Projection':<35} | {'N/A':>12} | {hybrid_param_info['component_breakdown']['fusion_projection_parameters']:>14,}")
print(f"{'  - Classification Head':<35} | {528:>12,} | {hybrid_param_info['component_breakdown']['classifier_parameters']:>14,}")
print("-" * 68)
ratio = hybrid_params / cnn_params
print(f"Complexity Multiplier: {ratio:.2f}x parameter increase over baseline 3D-CNN")
print("=" * 68)

ARCHITECTURAL COMPLEXITY COMPARISON
Component                           |       3D-CNN | CNN-ViT Hybrid
--------------------------------------------------------------------
Total Trainable Parameters          |       23,352 |        103,000
  - 3D-CNN Stream                   |       23,352 |         23,352
  - ViT Transformer Stream          |          N/A |         72,272
  - Fusion Projection               |          N/A |          6,336
  - Classification Head             |          528 |          1,040
--------------------------------------------------------------------
Complexity Multiplier: 4.41x parameter increase over baseline 3D-CNN


## 3. Regularization Strategy and Optimizer Formulation

With **103,000 parameters**, the hybrid model is susceptible to overfitting on minority land-cover classes (e.g., *Lettuce romaine 7wk* with ~1,079 pixels). 

### Regularization Mechanisms:
1. **Decoupled Weight Decay via AdamW**:
   Standard Adam implements $L_2$ regularization by adding $\lambda \mathbf{W}$ directly to the gradient $\mathbf{g}_t$. However, Adam's adaptive second-moment normalization $\sqrt{\hat{\mathbf{v}}_t}$ divides this penalty, reducing regularization for weights with large historic gradients.
   **AdamW** (Loshchilov & Hutter, 2019) decouples weight decay, applying it directly to the parameter update:
   $$\mathbf{W}_{t+1} \leftarrow \mathbf{W}_t - \eta_t \left( \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon} + \lambda \mathbf{W}_t \right)$$
   We configure $\lambda = 10^{-4}$, which restrains self-attention projections from memorizing training samples.

2. **Active Dropout in Fusion Head**:
   During `model.train()`, `Dropout(p=0.2)` randomly sets 20% of the fused representation activations to zero. This prevents co-adaptation and forces both the CNN stream and the ViT stream to learn robust, complementary features. During `model.eval()`, dropout is bypassed.

3. **Numerically Stable Multiclass Loss**:
   `nn.CrossEntropyLoss` operates directly on raw unnormalized logits $z_i$, computing $\mathcal{L} = -\log \left( \frac{e^{z_y}}{\sum_j e^{z_j}} \right)$ with the log-sum-exp trick.

## 4. Dataset Loading and Single-Batch Smoke Test

We load the PCA-reduced Salinas cube (15 spectral components explaining >99.7% spectral variance) and create stratified DataLoaders.

In [3]:
# 1. Load PCA-reduced Salinas cube and Ground Truth
raw_loader = SalinasLoader(raw_data_dir=PROJECT_ROOT / "data" / "raw")
dataset = raw_loader.load(allow_synthetic_fallback=True)
gt = dataset.gt

pca_path = PROJECT_ROOT / "data" / "processed" / "salinas_pca_reduced.npz"
if pca_path.is_file():
    pca_data = np.load(pca_path)
    cube = pca_data["reduced_cube"]
    print(f"Loaded PCA Cube: {pca_path} (Shape: {cube.shape})")
else:
    cube = dataset.cube[:, :, :15]

# 2. Construct DataLoaders
patch_size = 7
batch_size = 128

train_loader, val_loader, test_loader, meta = create_dataloaders(
    cube_or_dataset=cube,
    gt=gt,
    patch_size=patch_size,
    batch_size=batch_size,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    stratify=True,
    exclude_unlabeled=True,
)

print(f"Splits: Train={meta.train_samples:,} samples ({len(train_loader)} batches), "
      f"Val={meta.val_samples:,} samples ({len(val_loader)} batches)")

# 3. Single-batch optimization step smoke test
batch_x, batch_y = next(iter(train_loader))
smoke_model = HyperspectralHybridCNNViT(hybrid_config).to(device)
optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

smoke_bx = batch_x.to(device)
smoke_by = batch_y.to(device)

cnn_initial_p = next(smoke_model.cnn_branch.parameters()).clone()
vit_initial_p = next(smoke_model.vit_branch.parameters()).clone()

optimizer.zero_grad()
smoke_logits = smoke_model(smoke_bx)
loss = criterion(smoke_logits, smoke_by)
loss.backward()
optimizer.step()

cnn_updated_p = next(smoke_model.cnn_branch.parameters())
vit_updated_p = next(smoke_model.vit_branch.parameters())

print(f"\nSmoke Test Verification:")
print(f"  Input Batch Shape:   {tuple(smoke_bx.shape)}")
print(f"  Output Logits Shape: {tuple(smoke_logits.shape)}")
print(f"  Cross-Entropy Loss:  {loss.item():.4f}")
print(f"  CNN Stream Updated:  {not torch.allclose(cnn_initial_p, cnn_updated_p)}")
print(f"  ViT Stream Updated:  {not torch.allclose(vit_initial_p, vit_updated_p)}")
assert torch.isfinite(loss), "Loss must be finite"
print("[SUCCESS] Single optimization step passed cleanly across both branches!")

Loaded PCA Cube: C:\Users\abhis\Desktop\Tetraspectra\data\processed\salinas_pca_reduced.npz (Shape: (512, 217, 15))


Splits: Train=37,891 samples (297 batches), Val=8,118 samples (64 batches)



Smoke Test Verification:
  Input Batch Shape:   (128, 1, 15, 7, 7)
  Output Logits Shape: (128, 16)
  Cross-Entropy Loss:  3.1458
  CNN Stream Updated:  True
  ViT Stream Updated:  True
[SUCCESS] Single optimization step passed cleanly across both branches!


## 5. Model Training Run (2 Epochs on Full Salinas Dataset)

We execute 2 complete training epochs across all 37,891 training samples (297 batches per epoch) and evaluate generalization on all 8,118 validation samples.
Checkpoints are atomically written to `outputs/checkpoints/best_hybrid_cnn_vit.pt`.

In [4]:
checkpoint_dir = PROJECT_ROOT / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

train_config = TrainingConfig(
    epochs=2,
    lr=1e-3,
    weight_decay=1e-4,
    device=str(device),
    checkpoint_dir=str(checkpoint_dir),
    checkpoint_filename="best_hybrid_cnn_vit.pt",
    save_best_only=True,
    monitor_metric="val_loss",
    model_type="hybrid",
    seed=42,
)

model = HyperspectralHybridCNNViT(hybrid_config).to(device)

t0 = time.time()
trained_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    metadata=meta,
    model_config=hybrid_config,
)
total_time = time.time() - t0

print(f"\nTraining complete in {total_time:.2f} seconds.")
print(f"Best Epoch:           {history.best_epoch}")
print(f"Best Validation Loss: {history.best_val_loss:.4f}")
print(f"Best Validation Acc:  {history.best_val_acc * 100:.2f}%")
print(f"Saved Checkpoint:     {history.checkpoint_path}")


Training complete in 139.00 seconds.
Best Epoch:           2
Best Validation Loss: 0.0481
Best Validation Acc:  98.13%
Saved Checkpoint:     C:\Users\abhis\Desktop\Tetraspectra\outputs\checkpoints\best_hybrid_cnn_vit.pt


## 6. Training and Validation Diagnostic Curves

Visualizing loss descent and accuracy curves across training epochs diagnostics optimization dynamics and confirms that validation loss improves.

In [5]:
epochs_range = [ep.epoch for ep in history.epochs]
train_losses = [ep.train_loss for ep in history.epochs]
val_losses = [ep.val_loss for ep in history.epochs]
train_accs = [ep.train_acc * 100 for ep in history.epochs]
val_accs = [ep.val_acc * 100 for ep in history.epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss plot
ax1.plot(epochs_range, train_losses, marker="o", color="#1f77b4", lw=2, label="Train Loss")
ax1.plot(epochs_range, val_losses, marker="s", color="#ff7f0e", lw=2, linestyle="--", label="Val Loss")
ax1.axvline(history.best_epoch, color="#2ca02c", linestyle=":", label=f"Best Checkpoint (Ep {history.best_epoch})")
ax1.set_title("CNN-ViT Hybrid: Cross-Entropy Loss vs Epoch", fontsize=11, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_xticks(epochs_range)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Accuracy plot
ax2.plot(epochs_range, train_accs, marker="o", color="#1f77b4", lw=2, label="Train Accuracy")
ax2.plot(epochs_range, val_accs, marker="s", color="#ff7f0e", lw=2, linestyle="--", label="Val Accuracy")
ax2.axvline(history.best_epoch, color="#2ca02c", linestyle=":", label=f"Best Checkpoint (Ep {history.best_epoch})")
ax2.set_title("CNN-ViT Hybrid: Accuracy vs Epoch", fontsize=11, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_xticks(epochs_range)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

plt.tight_layout()

# Save figure to outputs/figures/
fig_dir = PROJECT_ROOT / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / "hybrid_training_curves.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved diagnostic curves to {fig_path}")
plt.show()

Saved diagnostic curves to C:\Users\abhis\Desktop\Tetraspectra\outputs\figures\hybrid_training_curves.png


C:\Users\abhis\AppData\Local\Temp\ipykernel_32528\1363170750.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Checkpoint Reload and Inference Verification

We reload the saved checkpoint into an uninitialized `HyperspectralHybridCNNViT` model instance, verify the stored metadata, and run forward inference on a validation batch.

In [6]:
checkpoint_path = Path(history.checkpoint_path)
print(f"Checkpoint File: {checkpoint_path}")
print(f"File Size:       {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")

# Instantiate fresh uninitialized model
fresh_hybrid = HyperspectralHybridCNNViT(hybrid_config).to(device)
payload = load_checkpoint(checkpoint_path, model=fresh_hybrid, device=device)

print(f"\nLoaded Checkpoint Metadata:")
print(f"  Model Architecture: {payload.get('model_architecture', 'HyperspectralHybridCNNViT')}")
print(f"  Epoch:              {payload['epoch']}")
print(f"  Val Loss:           {payload['val_loss']:.4f}")
print(f"  Val Accuracy:       {payload['val_acc'] * 100:.2f}%")
print(f"  Num Classes:        {payload['num_classes']}")
print(f"  Spectral Depth:     {payload['spectral_depth']}")
print(f"  Patch Size:         {payload['patch_size']}")

# Verify forward pass on loaded model
fresh_hybrid.eval()
with torch.no_grad():
    val_bx, val_by = next(iter(val_loader))
    val_bx, val_by = val_bx.to(device), val_by.to(device)
    val_logits = fresh_hybrid(val_bx)
    preds = val_logits.argmax(dim=-1)
    batch_acc = (preds == val_by).float().mean().item() * 100.0

print(f"\nFresh Model Forward Pass Verification:")
print(f"  Validation Logits Shape: {tuple(val_logits.shape)}")
print(f"  Sample Batch Accuracy:   {batch_acc:.2f}%")
print("[SUCCESS] Checkpoint successfully verified!")

Checkpoint File: C:\Users\abhis\Desktop\Tetraspectra\outputs\checkpoints\best_hybrid_cnn_vit.pt
File Size:       1.21 MB

Loaded Checkpoint Metadata:
  Model Architecture: HyperspectralHybridCNNViT
  Epoch:              2
  Val Loss:           0.0481
  Val Accuracy:       98.13%
  Num Classes:        16
  Spectral Depth:     15
  Patch Size:         7

Fresh Model Forward Pass Verification:
  Validation Logits Shape: (128, 16)
  Sample Batch Accuracy:   98.44%
[SUCCESS] Checkpoint successfully verified!


## 8. Summary & Architectural Lessons Learned

| Concept | Explanation |
|---|---|
| **Dual-Stream Gradient Flow** | Gradients from $\mathcal{L}_{\text{CE}}$ propagate through the classification head and fusion projection layer, splitting simultaneously into both the 3D-CNN branch and the ViT branch. |
| **Why Regularization is Critical** | The hybrid model contains **103,000 parameters** compared to **23,352** in the 3D-CNN baseline (4.4x increase). AdamW weight decay ($10^{-4}$) and fusion head dropout ($p=0.2$) are essential to prevent the transformer self-attention layers from memorizing training data. |
| **Decoupled Weight Decay** | Unlike standard $L_2$ regularization in Adam which divides weight penalties by running gradient moments, AdamW directly decays weights $\mathbf{W} \leftarrow \mathbf{W}(1 - \eta \lambda)$, providing uniform regularization across all attention layers. |
| **Separation of Checkpoints** | The hybrid checkpoint is stored at `outputs/checkpoints/best_hybrid_cnn_vit.pt`, preserving the Day 10 `best_3d_cnn.pt` baseline checkpoint intact for future comparative benchmarks. |

---

### What's Next (Day 15):
On **Day 15**, we will implement **Hyperparameter Optimization (HPO)** to systematically tune learning rates, weight decays, embedding dimensions, and dropout rates for this complex hybrid architecture.